# Setting up

In [22]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob
# from glob import glob

In [8]:
cell_type_colors = {
    # undefined - gray
    "Stromal_Undefined_Unknown": "#AAAAAA",  # gray
    "Unknown":"#AAAAAA",
    "Stromal_Undefined":"#AAAAAA",

    # tumor - red
    "Tumor": "#790000",           # dark red
    "Cycling_Tumor": "#D60000",   # bright red

    # endothelial - beige
    "Endothelial": "#A17569",     # muted beige

    # B - orange
    "B": "#C25A00",                # dark orange
    "Activated_B": "#FFA52F",     # orange

    # neutrophil - yellow
    "Neutrophil": "#FDF490",           # pastel yellow
    "Activated_Neutrophil": "#FFD700",  # bright yellow

    # macrophage - blue
    "Macrophage": "#0774D8",              # cobalt blue
    "Macrophage_CD163neg": "#0774D8",              # cobalt blue
    "Macrophage (CD163-)": "#0774D8",              # cobalt blue
    "Activated_Macrophage_CD163neg": "#9AE4FF",    # light sky blue
    "Macrophage_CD163pos": "#0000DD",              # royal blue
    "Macrophage (CD163+)": "#0000DD",              # royal blue
    "Activated_Macrophage_CD163pos": "#00ACC6",    # cyan blues

    # T - coral
    "T": "#EDB8B8",            # light pink
    "Activated_T": "#FF7266",  # coral red

    # CD4_T - green
    "CD4_T": "#004B00",                # dark green
    "Activated_CD4_T": "#97FF00",      # neon lime
    "Treg": "#005659",                 # teal
    "Activated_Treg": "#00FDCF",       # turquoise
    "Th1": "#5D7E66",                  # dusty green
    "Activated_Th1": "#018700",        # forest green

    # CD8_T - pink/purple
    "CD8_T": "#6B004F",                    # dark magenta
    "Activated_CD8_T": "#BF03B8",          # magenta
    "Exhausted_CD8": "#FF7ED1",            # pink
    "Exhausted CD8": "#FF7ED1",            # pink
    "Activated_Exhausted_CD8": "#EB0077",  # hot pink
    "Cytotoxic_NK": "#645474",             # muted purple
    "Cytotoxic NK": "#645474",             # muted purple
    "Activated_Cytotoxic_NK": "#8C3BFF"    # purple violet
}

marker_colors = {
    "E.cadherin": "#790000",           # dark red
    "CD31": "#A17569",     # muted beige
    "CD20": "#C25A00",                # dark orange
    "MPO": "#FDF490",           # pastel yellow
    "CD68": "#0774D8",              # cobalt blue
    "CD163": "#0000DD",              # royal blue
    "CD3e": "#EDB8B8",            # light pink
    "CD4": "#004B00",                # dark green
    "FOXP3": "#005659",                 # teal
    "TIM3": "#5D7E66",                  # dusty green
    "CD8": "#6B004F",                    # dark magenta
    "TOX": "#FF7ED1",            # pink
    "Granzyme.B": "#645474",             # muted purple
}

# Creating dataframes for all samples (do not rerun)

Here we create:
* `summary_df` - mean, median, and stdev per marker per sample
* `long_df` - marker exp prob per cell per marker per sample (for violin plots)
* `wide_df` - all marker exp prob dfs concatenated

Remember, marker expression probability is *not* threshold-dependent!

## summary_df, long_df (all samples)

In [13]:
import os
import pandas as pd
from glob import glob

base_dir = "/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types"
suffix = "*/*raw_arcsinh_marker_exp_prob.csv"  # only first subfolder level

csv_files = glob(os.path.join(base_dir, suffix), recursive=False)

summary_rows = []  # summary: mean, median, stdev
all_raw_rows = []  # all raw marker exp prob for violin plot

for file in csv_files:
    print(file)
    # skip_prefixes = ("endometrial", "detailed", "old", "test")
    parent_folder = os.path.basename(os.path.dirname(file))
    
    # if parent_folder.startswith(skip_prefixes):
    #     continue
    try:
        df = pd.read_csv(file)
        df.drop(columns=["X", "Y"], errors='ignore', inplace=True)

        sample_name = os.path.basename(os.path.dirname(file))  # parent folder
        sample_id = sample_name.split('_')[1]

        # get mean, median, stdev per marker
        for marker in df.columns:
            marker_values = df[marker].dropna()
            mean_val = marker_values.mean()
            median_val = marker_values.median()
            std_val = marker_values.std()

            summary_rows.append({
                "filename": sample_name,
                "sample_id": sample_id,
                "marker": marker,
                "mean": mean_val,
                "median": median_val,
                "stdev": std_val
            })

        # melt df to long format (marker, value) with sample info for raw exp prob df
        melted = df.melt(var_name="marker", value_name="value")
        melted["filename"] = sample_name
        melted["sample_id"] = sample_id

        all_raw_rows.append(melted)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# summary df
summary_df = pd.DataFrame(summary_rows)

# concatenated raw data df
long_df = pd.concat(all_raw_rows, ignore_index=True)

print(summary_df.head())
print(long_df.head())

/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_28873_raw_arcsinh/cervical_28873_raw_arcsinh_marker_exp_prob.csv
/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_04738_raw_arcsinh/cervical_04738_raw_arcsinh_marker_exp_prob.csv
/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_02433_raw_arcsinh/cervical_02433_raw_arcsinh_marker_exp_prob.csv
/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_07688_raw_arcsinh/cervical_07688_raw_arcsinh_marker_exp_prob.csv
/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_09002_raw_arcsinh/cervical_09002_raw_arcsinh_marker_exp_prob.csv
/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_34933_raw_arcsinh/cervical_34933_raw_arcsinh_marker_exp_prob.csv
/gpfs/data/proteomics/home/yb2612/results/celesta/detailed_cell_types/cervical_49411_raw_arcsinh/cervical_49411_raw_arcsinh_marker_exp_p

## wide_df (all samples)

In [14]:
# Create wide-format wide_df (each row = cell, each column = marker)
# Add unique cell_id per sample_id
wide_df_parts = []

for file in csv_files:
    parent_folder = os.path.basename(os.path.dirname(file))
    try:
        df = pd.read_csv(file)
        df.drop(columns=["X", "Y"], errors='ignore', inplace=True)

        sample_name = parent_folder
        sample_id = sample_name.split('_')[1]

        df["sample_id"] = sample_id

        # Add cell_id: e.g., "07688_Cell1", "07688_Cell2", ...
        df["cell_id"] = (
            df
            .groupby("sample_id")
            .cumcount()
            .add(1)
            .astype(str)
            .radd(sample_id + "_Cell")
        )

        wide_df_parts.append(df)

    except Exception as e:
        print(f"Error reading {file} for wide_df: {e}")

# Combine into one wide_df
wide_df = pd.concat(wide_df_parts, ignore_index=True)

# Reorder columns: cell_id, sample_id, [markers...]
marker_cols = [col for col in wide_df.columns if col not in ("cell_id", "sample_id")]
wide_df = wide_df[["cell_id", "sample_id"] + marker_cols]

print(wide_df.head())

       cell_id sample_id      CD31  E.cadherin      CD68  CD163       MPO  \
0  28873_Cell1     28873  0.571502    0.060849  0.715396    0.0  0.771662   
1  28873_Cell2     28873  0.433519    0.982813  0.823216    0.0  0.647071   
2  28873_Cell3     28873  0.174817    0.895753  0.846647    0.0  0.495428   
3  28873_Cell4     28873  0.182535    0.308473  0.466783    0.0  0.243098   
4  28873_Cell5     28873  0.410227    0.909573  0.764461    0.0  0.796262   

       CD20      CD3e       CD8  Granzyme.B       TOX       CD4     FOXP3  \
0  0.373763  0.428196  0.000000    0.118951  0.083003  0.459578  0.543199   
1  0.805770  0.470395  0.000000    0.804407  0.769293  0.442903  0.757446   
2  0.727329  0.374826  0.000000    0.755836  0.659265  0.385593  0.747067   
3  0.249756  0.106070  0.000000    0.146604  0.048669  0.236750  0.343430   
4  0.675112  0.491498  0.013215    0.692156  0.633338  0.462929  0.730392   

       TIM3  
0  0.388348  
1  0.697648  
2  0.571555  
3  0.251292  
4  0

In [15]:
print(summary_df.shape)
print(long_df.shape)
print(wide_df.shape)

(182, 6)
(83398627, 4)
(6415279, 15)


In [29]:
print("exporting summary_df")
summary_df.to_csv("/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/summary_expression_probability.csv",index=False)
print("done")

print("exporting long_df")
long_df.to_csv("/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/all_expression_probability_long.csv",index=False)
print("done")

print("exporting wide_df")
wide_df.to_csv("/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/all_expression_probability_wide.csv",index=False)
print("done")

exporting summary_df
done
exporting long_df
done
exporting wide_df
done


# Loading dataframes for all samples

In [59]:
# summary_df = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/summary_expression_probability.csv")
# long_df = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/all_expression_probability_long.csv")
wide_df = pd.read_csv("/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/all_expression_probability_wide.csv")

In [62]:
# pad sample_id col to 5 digits with leading 0s
wide_df["sample_id"] = wide_df["sample_id"].astype(str).str.zfill(5)
wide_df

,cell_id,sample_id,CD31,E.cadherin,CD68,CD163,MPO,CD20,CD3e,CD8,Granzyme.B,TOX,CD4,FOXP3,TIM3
0,28873_Cell1,28873,0.571502,0.060849,0.715396,0.000000,0.771662,0.373763,0.428196,0.000000,0.118951,0.083003,0.459578,0.543199,0.388348
1,28873_Cell2,28873,0.433519,0.982813,0.823216,0.000000,0.647071,0.805770,0.470395,0.000000,0.804407,0.769293,0.442903,0.757446,0.697648
2,28873_Cell3,28873,0.174817,0.895753,0.846647,0.000000,0.495428,0.727329,0.374826,0.000000,0.755836,0.659265,0.385593,0.747067,0.571555
3,28873_Cell4,28873,0.182535,0.308473,0.466783,0.000000,0.243098,0.249756,0.106070,0.000000,0.146604,0.048669,0.236750,0.343430,0.251292
4,28873_Cell5,28873,0.410227,0.909573,0.764461,0.000000,0.796262,0.675112,0.491498,0.013215,0.692156,0.633338,0.462929,0.730392,0.647223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6415274,08153_Cell103616,08153,0.050292,0.089653,0.051213,0.000000,0.953748,0.354799,0.251470,0.000000,0.000000,0.000000,0.052633,0.008000,0.032631
6415275,08153_Cell103617,08153,0.029455,0.013938,0.028490,0.000000,0.000000,0.154803,0.258845,0.000000,0.000000,0.000000,0.013888,0.005363,0.021337
6415276,08153_Cell103618,08153,0.016766,0.001077,0.046210,0.002532,0.001144,0.056078,0.334356,0.000000,0.000960,0.002185,0.011211,0.019191,0.009835
6415277,08153_Cell103619,08153,0.025634,0.517115,0.037648,0.000000,0.000000,0.188245,0.147783,0.000000,0.000000,0.000000,0.170419,0.003195,0.016895


# Adding final cell type to wide_df (per sample)

Remember, final cell type annotations *are* threshold-dependent. For each set of thresholds tested, append a column to `wide_df` with corresponding cell type assignments.

In [63]:
# select samples by commenting out
samples = [
    # ### small
    # "10103",
    "28873",
    "34933",
    "49411",
    "39367",

    # ### medium
    # "08153",
    # "09002",
    # "02433",
    # "07688",
    # "00438",
    # "04738",

    # ### large
    # "07291",
    # "00862",
    # "10285"
] 

In [64]:
labels = {}

for sample in samples:
    print(sample)
    
    labels[sample] = []
    
    # Folder containing the CELlESTA results
    celesta_dir = f"/gpfs/home/yb2612/yb2612_fenyo/results/celesta/detailed_cell_types/cervical_{sample}_raw_arcsinh"
    
    # Find all final_cell_type_assignment.csv files
    final_type_files = sorted(glob.glob(os.path.join(celesta_dir, "*final_cell_type_assignment.csv")))
    
    wide_df_trimmed = wide_df[wide_df["sample_id"]==sample].copy().reset_index(drop=True)
    # print(wide_df.head())
    # print(wide_df_trimmed.head())
    
    # Loop through files and add each Final cell type column to wide_df
    for file in final_type_files:
        # Extract short unique label from filename for column naming
        label = os.path.basename(file).replace(f"cervical_{sample}_raw_arcsinh_", "").replace("_final_cell_type_assignment.csv", "")
        print(label)
    
        labels[sample].append(label)
    
        # Load final cell type column
        df = pd.read_csv(file)
    
        if "Final cell type" not in df.columns:
            print(f"Skipping {file}: 'Final cell type' column not found.")
            continue
    
        # Add to wide_df
        wide_df_trimmed[label] = df["Final cell type"]
    
    # Save updated wide_df
    output_path = f"/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/{sample}_expression_probability_wide_celltypes.csv"
    wide_df_trimmed.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

28873
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.6_0.5_0.5_0.5_0.5_0.5_0.5_0.5
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.7_0.5_0.5_0.5_0.5_0.5_0.5_0.5
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_default
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.6_0.5_0.5_0.5_0.5_0.5_0.5_0.5
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.7_0.5_0.5_0.5_0.5_0.5_0.5_0.5
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.8_0.5_0.5_0.5_0.5_0.5_0.5_0.5
high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_default
high_anchor_default_high_iter_default
Saved: /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/28873_expression_probability_w

# Mean expression probability (violin plots)

Violin plots of mean expression probability per sample per marker.

In [44]:
output_dir = "/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability"  # <-- set your output folder here
os.makedirs(output_dir, exist_ok=True)

for marker_to_plot in long_df["marker"].unique():
    plot_df = long_df[long_df["marker"] == marker_to_plot]

    plt.figure(figsize=(12, 6))

    # get color from dict or fallback to gray
    color = marker_colors.get(marker_to_plot, "#AAAAAA")

    ax = sns.violinplot(
        x="sample_id",
        y="value",
        data=plot_df,
        density_norm="width",
        inner="box",
        color=color,
        linewidth=1.2
    )

    # add mean values from summary_df as labels on top
    group_max = plot_df.groupby("sample_id")["value"].max()
    group_means = summary_df[(summary_df["marker"] == marker_to_plot)].set_index("sample_id")["mean"]

    for i, sample in enumerate(group_max.index):
        max_val = group_max[sample]
        mean_val = group_means.loc[sample]
        if hasattr(mean_val, 'values'):
            mean_val = mean_val.values[0]
        ax.text(i, max_val + 0.05 * max_val, f"{mean_val:.2f}",
                ha='center', va='bottom', fontsize=10, color='black')

    plt.title(f"{marker_to_plot} Expression Probability (arcsinh)")
    plt.ylabel(f"{marker_to_plot} Expression Probability")
    plt.xlabel("Sample")
    plt.tight_layout()

    # save
    filename = f"{marker_to_plot}_exp_prob_violinplot.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300)
    plt.close() 

    print(f"Saved plot for {marker_to_plot} to {save_path}")

Saved plot for CD31 to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/CD31_exp_prob_violinplot.png
Saved plot for E.cadherin to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/E.cadherin_exp_prob_violinplot.png
Saved plot for CD68 to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/CD68_exp_prob_violinplot.png
Saved plot for CD163 to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/CD163_exp_prob_violinplot.png
Saved plot for MPO to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/MPO_exp_prob_violinplot.png
Saved plot for CD20 to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/CD20_exp_prob_violinplot.png
Saved plot for CD3e to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/CD3e_exp_prob_violinplot.png
Saved plot for CD8 to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expressio

## Fixing specific plots

In [50]:
output_dir = "/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability"  # <-- set your output folder here
os.makedirs(output_dir, exist_ok=True)
            
marker_to_plot = "Granzyme.B"
plot_df = long_df[long_df["marker"] == marker_to_plot]

plt.figure(figsize=(12, 6))

# get color from dict or fallback to gray
color = marker_colors.get(marker_to_plot, "#AAAAAA")

ax = sns.violinplot(
    x="sample_id",
    y="value",
    data=plot_df,
    density_norm="width",
    inner="box",
    color=color,
    linewidth=1.2
)

# add mean values from summary_df as labels on top
group_max = plot_df.groupby("sample_id")["value"].max()
group_means = summary_df[(summary_df["marker"] == marker_to_plot)].set_index("sample_id")["mean"]

for i, sample in enumerate(group_max.index):
    max_val = group_max[sample]
    mean_val = group_means.loc[sample]
    if hasattr(mean_val, 'values'):
        mean_val = mean_val.values[0]
    ax.text(i, max_val + 0.08 * max_val, f"{mean_val:.2f}",  # change value added to max_val to adjust label height
            ha='center', va='bottom', fontsize=10, color='black')

plt.title(f"{marker_to_plot} Expression Probability (arcsinh)")
plt.ylabel(f"{marker_to_plot} Expression Probability")
plt.xlabel("Sample")
plt.tight_layout()

# save
filename = f"{marker_to_plot}_exp_prob_violinplot.png"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, dpi=300)
plt.close() 

print(f"Saved plot for {marker_to_plot} to {save_path}")

Saved plot for Granzyme.B to /gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability/Granzyme.B_exp_prob_violinplot.png


# Pairwise marker expression probability (scatter plots)

In [30]:
cell_type_colors_dark = {
    'Unknown': '#808080',                       # gray
    'Stromal_Undefined': '#A9A9A9',             # dark gray
    'Stromal cells (undefined)': '#A9A9A9',     # dark gray
    'Tumor': '#A42A2A',                         # crimson
    'Tumor cells': '#A42A2A',                   # crimson
    'Endothelial': '#33FD02',                   # green
    'Endothelial cells': '#33FD02',             # green
    'Neutrophil': '#34FEFF',                    # cyan
    'Neutrophils': '#34FEFF',                   # cyan
    'Macrophage': '#FB22FF',                    # magenta
    'Macrophage (CD163-)': '#FB22FF',           # magenta
    'Macrophages (CD163-)': '#FB22FF',          # magenta
    'Macrophage (CD163+)': '#FF80FF',           # light magenta
    'Macrophages (CD163+)': '#FF80FF',          # light magenta
    'CD8_T': '#FFFE04',                         # yellow
    'Cytotoxic T cells': '#FFFE04',             # yellow
    'T': '#008B8B',                             # dark cyan
    'T cells (other)': '#008B8B',               # dark cyan
    'CD4_T': '#FC8001',                         # orange
    'Helper T cells': '#FC8001',                # orange
    'B': '#FFFFFF',                             # white
    'B cells': '#FFFFFF',                       # white
    'Cytotoxic NK': '#B7FFFA',                  # pastel cyan
    'Exhausted CD8': '#FFC1CB',                 # pastel pink
    'Treg': '#FFC067',                          # pastel orange
    'Th1': '#FFEE8C',                           # pastel yellow
}

# Set global style with large font, black background, and minimal grid
sns.set_theme(
    style="white",  # start clean
    rc={
        "axes.facecolor": "black",
        "figure.facecolor": "black",
        "grid.color": "#444444",   # faint grid
        "axes.edgecolor": "white",
        "axes.labelcolor": "white",
        "xtick.color": "white",
        "ytick.color": "white",
        "legend.facecolor": "black",
        "legend.edgecolor": "white",
        "legend.labelcolor": "white",
        "font.size": 18,
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
    }
)

base_path = "/gpfs/home/yb2612/yb2612_fenyo/results/celesta/marker_expression_probability"

In [ ]:
# select samples by commenting out
samples = [
    # ### small
    # "10103",
    "28873",
    "34933",
    "49411",
    "39367",

    # ### medium
    # "08153",
    # "09002",
    # "02433",
    # "07688",
    # "00438",
    # "04738",

    # ### large
    # "07291",
    # "00862",
    # "10285"
] 

In [65]:
for sample in samples:
    print(sample)
    # Load wide format cell types
    wide_df_path = os.path.join(base_path, f"{sample}_expression_probability_wide_celltypes.csv")
    wide_df = pd.read_csv(wide_df_path)

    # Loop through each cell type assignment column
    for label in wide_df.columns:
        if label.startswith("high"):
            print(f"Plotting sample {sample}, label {label}")
    
            df_plot = wide_df.copy()
            df_plot["Final.cell.type"] = df_plot[label]
    
            # Filter for Tumor and B
            df_filtered = df_plot[
                (df_plot["Final.cell.type"] == "Tumor") |
                (df_plot["Final.cell.type"].str.contains("B", na=False))
            ]
    
            if df_filtered.empty:
                print(f"Skipping {label} — no Tumor or B cells")
                continue
    
            plt.figure(figsize=(20, 15))
            ax = sns.scatterplot(
                data=df_filtered,
                x="CD20",
                y="E.cadherin",
                hue="Final.cell.type",
                palette=cell_type_colors_dark,
                s=12,
                edgecolor=None,
                alpha=0.9
            )
    
            ax.set_xticks(np.arange(0, df_filtered["CD20"].max() + 0.1, 0.1))
            ax.set_yticks(np.arange(0, df_filtered["E.cadherin"].max() + 0.1, 0.1))
            ax.grid(True, linestyle='--', linewidth=0.5, color="#444444", alpha=0.3)
            ax.set_axisbelow(True)
    
            ax.set_title(f"CD20 vs E.cadherin - {sample}", color='white')
            ax.set_xlabel("CD20 Expression Probability (arcsinh)")
            ax.set_ylabel("E.cadherin Expression Probability (arcsinh)")
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=18, title_fontsize=18, markerscale=2.5)
    
            output_file = os.path.join(
                base_path,
                f"CD20vEcadherin_{sample}_{label}.png"
            )
            plt.tight_layout()
            plt.savefig(output_file, dpi=300, facecolor='black')
            plt.close()
            print("saved")

28873
Plotting sample 28873, label high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.6_0.5_0.5_0.5_0.5_0.5_0.5_0.5
saved
Plotting sample 28873, label high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.7_0.5_0.5_0.5_0.5_0.5_0.5_0.5
saved
Plotting sample 28873, label high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_default
saved
Plotting sample 28873, label high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.6_0.5_0.5_0.5_0.5_0.5_0.5_0.5
saved
Plotting sample 28873, label high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.7_0.5_0.5_0.5_0.5_0.5_0.5_0.5
saved
Plotting sample 28873, label high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.8_0.5_0.5_0.5_0.5_0.5_0.5_0.5
saved
Plotting sample 28873, label